In [1]:
import pandas as pd
import os

In [2]:
def rewrite_all_results(root, ban_list, fnum, fused):
    top = False
    if top == True:
        num = 12
    all_results = []
    counter = 0
    if fused:
        for disease in os.listdir(root):
            if disease.startswith('ICD') and disease.split('_')[-1].split('.')[0] not in ban_list:
                result_df = pd.read_csv(os.path.join(root,disease))
                if len(result_df) == fnum:
                    counter += 1
                    mean_df = result_df.groupby(['method','para'])[['top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30']].mean().reset_index()
                    # Add disease information
                    mean_df['disease'] = disease.split('.')[0]
                    # Append to all_results list
                    all_results.append(mean_df)
                else:
                    print(disease, 'not enough feature')
            if top == True:
                if counter == num:
                    break
        # Concatenate all results into a single DataFrame
        final_result = pd.concat(all_results, ignore_index=True)
        if top == True:
            final_result.to_csv(os.path.join(root,f'all_disease_{num}.csv'),index=False)
        else:
            final_result.to_csv(os.path.join(root,'all_disease.csv'),index=False)
        return final_result
    else:
        for disease in os.listdir(root):
            if disease.startswith('ICD'):
                counter += 1
                result_df = pd.read_csv(os.path.join(root,disease))
                mean_df = result_df.groupby(['method'])[['top_recall_25','top_recall_300','top_recall_10%', 'top_precision_10%', 'max_precision_10%','top_recall_30%', 'top_precision_30%', 'max_precision_30%','pm_0.5%','pm_1%','pm_5%','pm_10%','pm_15%','pm_20%','pm_25%','pm_30%','auroc',"rank_ratio",'bedroc_1','bedroc_5','bedroc_10','bedroc_30']].mean().reset_index()
                # Add disease information
                mean_df['disease'] = disease.split('.')[0]
                # Append to all_results list
                all_results.append(mean_df)
            if top == True:
                if counter == num:
                    break
        # Concatenate all results into a single DataFrame
        final_result = pd.concat(all_results, ignore_index=True)
        if top == True:
            final_result.to_csv(os.path.join(root,f'all_disease_{num}.csv'),index=False)
        else:
            final_result.to_csv(os.path.join(root,'all_disease.csv'),index=False)
        return final_result

def create_summary(results,col_name, fused):
    # Create an empty list to store results
    summary_list = []
    if fused == False:
        # Grouping by 'method' and calculating mean and std for selected metrics
        for method, subdf in results.groupby(col_name):
            mean_values = subdf.iloc[:, 1:23].mean()  # Selecting all metric columns
            std_values = subdf.iloc[:, 1:23].std()
            summary_list.append({
                'method': method,
                'top_recall_25_mean': mean_values['top_recall_25'], 'top_recall_25_std': std_values['top_recall_25'],
                'top_recall_300_mean': mean_values['top_recall_300'], 'top_recall_300_std': std_values['top_recall_300'],
                'top_recall_10_mean': mean_values['top_recall_10%'], 'top_recall_10_std': std_values['top_recall_10%'],
                'top_precision_10_mean': mean_values['top_precision_10%'], 'top_precision_10_std': std_values['top_precision_10%'],
                'max_precision_10_mean': mean_values['max_precision_10%'], 'max_precision_10_std': std_values['max_precision_10%'],
                'top_recall_30_mean': mean_values['top_recall_30%'], 'top_recall_10_std': std_values['top_recall_30%'],
                'top_precision_30_mean': mean_values['top_precision_30%'], 'top_precision_10_std': std_values['top_precision_30%'],
                'max_precision_30_mean': mean_values['max_precision_30%'], 'max_precision_10_std': std_values['max_precision_30%'],
                'pm_0.5%': mean_values['pm_0.5%'], 'pm_0.5%_std': std_values['pm_0.5%'],
                'pm_1%': mean_values['pm_1%'], 'pm_1%_std': std_values['pm_1%'],
                'pm_5%': mean_values['pm_5%'], 'pm_5%_std': std_values['pm_5%'],
                'pm_10%': mean_values['pm_10%'], 'pm_10%_std': std_values['pm_10%'],
                'pm_15%': mean_values['pm_15%'], 'pm_15%_std': std_values['pm_15%'],            
                'pm_20%': mean_values['pm_20%'], 'pm_20%_std': std_values['pm_20%'],
                'pm_25%': mean_values['pm_25%'], 'pm_1%_std': std_values['pm_25%'],
                'pm_30%': mean_values['pm_30%'], 'pm_30%_std': std_values['pm_30%'],
                'auroc_mean': mean_values['auroc'], 'auroc_std': std_values['auroc'],
                'rank_ratio_mean': mean_values['rank_ratio'], 'rank_ratio_std': std_values['rank_ratio'],
                'bedroc_1_mean': mean_values['bedroc_1'], 'bedroc_1_std': std_values['bedroc_1'],
                'bedroc_5_mean': mean_values['bedroc_5'], 'bedroc_5_std': std_values['bedroc_5'],
                'bedroc_10_mean': mean_values['bedroc_10'], 'bedroc_10_std': std_values['bedroc_10'],
                'bedroc_30_mean': mean_values['bedroc_30'], 'bedroc_30_std': std_values['bedroc_30'],
            })

        # Convert the list of dictionaries into a DataFrame
        summary_df = pd.DataFrame(summary_list)
    else:
                # Grouping by 'method' and calculating mean and std for selected metrics
        for paras, subdf in results.groupby(col_name):
            mean_values = subdf.iloc[:, 2:24].mean()  # Selecting all metric columns
            std_values = subdf.iloc[:, 2:24].std()
            summary_list.append({
                'method': paras[0],
                'para': paras[1],
                'top_recall_25_mean': mean_values['top_recall_25'], 'top_recall_25_std': std_values['top_recall_25'],
                'top_recall_300_mean': mean_values['top_recall_300'], 'top_recall_300_std': std_values['top_recall_300'],
                'top_recall_10_mean': mean_values['top_recall_10%'], 'top_recall_10_std': std_values['top_recall_10%'],
                'top_precision_10_mean': mean_values['top_precision_10%'], 'top_precision_10_std': std_values['top_precision_10%'],
                'max_precision_10_mean': mean_values['max_precision_10%'], 'max_precision_10_std': std_values['max_precision_10%'],
                'top_recall_30_mean': mean_values['top_recall_30%'], 'top_recall_10_std': std_values['top_recall_30%'],
                'top_precision_30_mean': mean_values['top_precision_30%'], 'top_precision_10_std': std_values['top_precision_30%'],
                'max_precision_30_mean': mean_values['max_precision_30%'], 'max_precision_10_std': std_values['max_precision_30%'],
                'pm_0.5%': mean_values['pm_0.5%'], 'pm_0.5%_std': std_values['pm_0.5%'],
                'pm_1%': mean_values['pm_1%'], 'pm_1%_std': std_values['pm_1%'],
                'pm_5%': mean_values['pm_5%'], 'pm_5%_std': std_values['pm_5%'],
                'pm_10%': mean_values['pm_10%'], 'pm_10%_std': std_values['pm_10%'],
                'pm_15%': mean_values['pm_15%'], 'pm_15%_std': std_values['pm_15%'],            
                'pm_20%': mean_values['pm_20%'], 'pm_20%_std': std_values['pm_20%'],
                'pm_25%': mean_values['pm_25%'], 'pm_1%_std': std_values['pm_25%'],
                'pm_30%': mean_values['pm_30%'], 'pm_30%_std': std_values['pm_30%'],
                'auroc_mean': mean_values['auroc'], 'auroc_std': std_values['auroc'],
                'rank_ratio_mean': mean_values['rank_ratio'], 'rank_ratio_std': std_values['rank_ratio'],
                'bedroc_1_mean': mean_values['bedroc_1'], 'bedroc_1_std': std_values['bedroc_1'],
                'bedroc_5_mean': mean_values['bedroc_5'], 'bedroc_5_std': std_values['bedroc_5'],
                'bedroc_10_mean': mean_values['bedroc_10'], 'bedroc_10_std': std_values['bedroc_10'],
                'bedroc_30_mean': mean_values['bedroc_30'], 'bedroc_30_std': std_values['bedroc_30'],
            })

        # Convert the list of dictionaries into a DataFrame
        summary_df = pd.DataFrame(summary_list)
    return summary_df

In [5]:
from IPython.display import display, HTML

In [3]:
def show_table(root, ban_list, fnum, fused, input_weights):
    if fused:
        final_result = rewrite_all_results(root, ban_list, fnum, fused=True)
        if input_weights:
            weight_df = final_result.copy()
            weight_df[['para', 'weights']] = weight_df['para'].str.split('-', expand=True)
            all_sum = create_summary(weight_df.drop(columns='weights'), ['method', 'para'], fused=True)
            show_df = all_sum.sort_values(by='method', ascending=True) \
                .loc[:, all_sum.columns.str.contains(
                    'method|para|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|rank_ratio', case=False)] \
                .round(3).rename(columns=lambda x: x.replace('_mean', ''))
            display(HTML(show_df.to_html(index=False).replace('<table', '<table style="font-size:13px; white-space:nowrap;"')))
            return weight_df, show_df
        else:
            all_sum = create_summary(final_result, ['method', 'para'], fused=True)
            show_df = all_sum.sort_values(by='method', ascending=True) \
                .loc[:, all_sum.columns.str.contains(
                    'method|para|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean| rank_ratio', case=False)] \
                .round(3).rename(columns=lambda x: x.replace('_mean', ''))
            display(HTML(show_df.to_html(index=False).replace('<table', '<table style="font-size:13px; white-space:nowrap;"')))
            return final_result, show_df
    else:
        final_result = rewrite_all_results(root, fused=False)
        all_sum = create_summary(final_result, 'method', fused=False)
        if 'random_pos_negative_bagging' in all_sum['method'].unique():
            method_order = ['random_negativeauroc', 'random_negative_bagging', 'random_pos_negative_bagging']
            all_sum['method'] = pd.Categorical(all_sum['method'], categories=method_order, ordered=True)
        
        show_df = all_sum.sort_values(by='method', ascending=True) \
            .loc[:, all_sum.columns.str.contains(
                'method|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|rank_ratio', case=False)] \
            .round(3).rename(columns=lambda x: x.replace('_mean', ''))
        
        display(HTML(show_df.to_html(index=False).replace('<table', '<table style="font-size:13px; white-space:nowrap;"')))
        return final_result, show_df

    

icd_dict = {
    'Certain infectious and parasitic diseases': ['A00','B99'],
    'Neoplasms': ['C00','D48'],
    'Diseases of the blood and blood-forming organs and certain disorders involving the immune mechanism': ['D50','D89'],
    'Endocrine, nutritional and metabolic diseases': ['E00','E90'],
    'Mental and behavioural disorders': ['F00','F99'],
    'Diseases of the nervous system': ['G00','G99'],
    'Diseases of the eye and adnexa': ['H00','H59'],
    'Diseases of the ear and mastoid process': ['H60','H95'],
    'Diseases of the circulatory system': ['I00','I99'],
    'Diseases of the respiratory system': ['J00','J99'],
    'Diseases of the digestive system': ['K00','K93'],
    'Diseases of the skin and subcutaneous tissue': ['L00','L99'],
    'Diseases of the musculoskeletal system and connective tissue': ['M00','M99'],
    'Diseases of the genitourinary system': ['N00','N99']
}

def find_disease_category(icd_code):
    icd_num = icd_code.split('_')[1]  # Extract ICD-10 code
    # print(icd_num)
    icd_letter = icd_num[0]  # Extract first letter (C, D, etc.)
    icd_number = int(icd_num[1:])  # Extract numeric part

    for category, (start, end) in icd_dict.items():
        start_letter, start_num = start[0], int(start[1:])
        end_letter, end_num = end[0], int(end[1:])

        if start_letter <= icd_letter <= end_letter:  # Ensure it's within the letter range
            if start_letter == icd_letter and start_num <= icd_number:
                return category
            if end_letter == icd_letter and icd_number <= end_num:
                return category
            if start_letter < icd_letter < end_letter:
                return category  # Covers ranges like C00-D48
        
    return 'Unknown Category'

def disease_catygory(results):
    mapped_results = {icd: find_disease_category(icd) for icd in results['disease']}
    results['category'] = results['disease'].map(mapped_results)

    collected_dfs = []
    disease_num = dict()
    for category in results['category'].unique().tolist():
        subdf = results[results['category'] == category].copy()
        sum_df = create_summary(subdf, 'method', fused=False)
        sum_df['category'] = category
        collected_dfs.append(sum_df)
        disease_num[category] = len(subdf) / 2

    final_df = pd.concat(collected_dfs, ignore_index=True)

    category_order = final_df.groupby('category', observed=True)['auroc_mean'].mean().sort_values(ascending=False).index.tolist()
    final_df['category'] = pd.Categorical(final_df['category'], categories=category_order, ordered=True)

    show_df = final_df.sort_values(by=['category', 'auroc_mean'], ascending=[True, False]) \
        .loc[:, final_df.columns.str.contains('method|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|category', case=False)] \
        .round(3) \
        .rename(columns=lambda x: x.replace('_mean', ''))

    # Highlight specific method
    def highlight_method(val):
        if isinstance(val, str) and val == 'random_pos_negative_bagging':
            return '<span style="color:red; font-weight:bold;">random_pos_negative_bagging</span>'
        return val

    styled_df = show_df.copy()
    if 'method' in styled_df.columns:
        styled_df['method'] = styled_df['method'].apply(highlight_method)

    # Assign background colors per category
    category_colors = {}
    base_colors = ['#ffdddd', "#dbf7db"]
    for i, cat in enumerate(category_order):
        category_colors[cat] = base_colors[i % len(base_colors)]

    def row_style(row):
        color = category_colors.get(row['category'], '#ffffff')
        return [f'background-color: {color}'] * len(row)


    # Display styled table
    from IPython.display import display, HTML
    display(HTML(styled_df.style.apply(row_style, axis=1).to_html(escape=False)))


# def disease_catygory_fused(results):
#     mapped_results = {icd: find_disease_category(icd) for icd in results['disease']}
#     results['category'] = results['disease'].map(mapped_results)

#     collected_dfs = []
#     disease_num = dict()
#     for category in results['category'].unique().tolist():
#         subdf = results[results['category'] == category].copy()
#         subdf = subdf.drop(columns='weights')
#         sum_df = create_summary(subdf, ['method', 'para'], fused=True)
#         sum_df['category'] = category
#         collected_dfs.append(sum_df)
#         disease_num[category] = len(subdf) / 2

#     final_df = pd.concat(collected_dfs, ignore_index=True)

#     category_order = final_df.groupby('category', observed=True)['auroc_mean'].mean().sort_values(ascending=False).index.tolist()
#     final_df['category'] = pd.Categorical(final_df['category'], categories=category_order, ordered=True)

#     show_df = final_df.sort_values(by=['category', 'auroc_mean'], ascending=[True, False]) \
#         .loc[:, final_df.columns.str.contains('method|para|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|category', case=False)] \
#         .round(3) \
#         .rename(columns=lambda x: x.replace('_mean', ''))

#     # Define the custom order for 'feature'
#     feature_order = ['ppi_2019', 'bioconcept', 'esm2', 'uniport', 
#                     'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused']

#     # Convert 'feature' to a categorical type with that order
#     show_df['para'] = pd.Categorical(show_df['para'], categories=feature_order, ordered=True)

#     # Now sort by 'disease' first, then 'feature' by the custom order
#     show_df = show_df.sort_values(by=['category', 'para'])
    
#     # Define alternating background colors per disease
#     base_colors = ['#ffdddd', '#dbf7db']
#     target_cols = ['auroc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30', 'weights']

#     # Build mapping for background colors per disease
#     diseases = show_df['disease'].unique()
#     disease_color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(diseases)}

#     # Function to create full styling DataFrame
#     def combined_style(df):
#         styles = pd.DataFrame('', index=df.index, columns=df.columns)

#         # Add background color row-wise
#         for idx, row in df.iterrows():
#             bg_color = disease_color_map[row['disease']]
#             styles.loc[idx, :] = f'background-color: {bg_color};'

#         # Highlight max in each group & each target column
#         for disease, group in df.groupby('disease'):
#             for col in target_cols:
#                 max_val = group[col].max()
#                 max_indices = group[group[col] == max_val].index
#                 for idx in max_indices:
#                     styles.loc[idx, col] += ' color: red; font-weight: bold;'
        
#         return styles

#     # Apply combined style
#     styled_df = show_df.style.apply(combined_style, axis=None)
#     return styled_df

In [6]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2017_nn_uni'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019, all_avg_df = show_table(root,ban_list=[],fnum=8,fused = True,input_weights = True)

ICD10_N97.csv not enough feature


method,para,auroc,rank_ratio,rank_ratio_std,bedroc_1,bedroc_5,bedroc_10,bedroc_30
random_negative,DL_early,0.669,0.331,0.191,0.048,0.134,0.202,0.368
random_negative,DL_later,0.751,0.249,0.190,0.083,0.226,0.316,0.495
random_negative,DL_mid,0.778,0.222,0.186,0.123,0.273,0.365,0.542
random_negative,DL_ppi_2017_dw_80,0.758,0.242,0.190,0.103,0.248,0.334,0.510
random_negative,DL_uniport_esm,0.614,0.386,0.182,0.027,0.086,0.139,0.293
random_negative,DL_uniport_exp,0.584,0.416,0.193,0.014,0.064,0.113,0.262
random_negative,DL_uniport_ppi_2017,0.752,0.248,0.182,0.091,0.201,0.291,0.488
random_negative,DL_uniport_seq,0.650,0.350,0.193,0.028,0.116,0.179,0.345


In [7]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_nn_uni'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019, all_avg_df = show_table(root,ban_list=[],fnum=8,fused = True,input_weights = True)

method,para,auroc,rank_ratio,rank_ratio_std,bedroc_1,bedroc_5,bedroc_10,bedroc_30
random_negative,DL,0.998,0.002,0.002,0.761,0.940,0.969,0.989
random_negative,DL_early,0.676,0.324,0.188,0.050,0.135,0.204,0.371
random_negative,DL_later,0.764,0.236,0.172,0.092,0.230,0.314,0.491
random_negative,DL_mid,0.729,0.271,0.192,0.108,0.230,0.300,0.460
random_negative,DL_ppi_2019_dw_40,0.753,0.247,0.188,0.079,0.220,0.302,0.478
random_negative,DL_uniport_bio,0.694,0.306,0.190,0.056,0.154,0.225,0.391
random_negative,DL_uniport_esm,0.620,0.380,0.176,0.018,0.070,0.126,0.293
random_negative,DL_uniport_ppi_2019,0.724,0.276,0.173,0.053,0.180,0.253,0.426
random_negative,DL_uniport_seq,0.639,0.361,0.197,0.022,0.084,0.145,0.315


In [38]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2017_cv_rank_save_auc0.8'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019, all_avg_df = show_table(root,ban_list=[],fnum=8,fused = True,input_weights = True)

ICD10_C50.csv not enough feature
ICD10_F90.csv not enough feature
ICD10_F72.csv not enough feature
ICD10_G91.csv not enough feature
ICD10_G24.csv not enough feature


method,para,auroc,rank_ratio,rank_ratio_std,bedroc_1,bedroc_5,bedroc_10,bedroc_30
random_negative,early_fusion,0.662,0.338,0.194,0.059,0.141,0.206,0.369
random_negative,geo_fused,0.817,0.184,0.172,0.169,0.316,0.419,0.607
random_negative,linear_fused,0.818,0.183,0.170,0.171,0.321,0.424,0.610
random_negative,ppi_2017_dw_80,0.823,0.177,0.169,0.165,0.331,0.436,0.617
random_negative,uniport_esm,0.658,0.342,0.194,0.055,0.139,0.204,0.367
random_negative,uniport_exp,0.666,0.334,0.168,0.016,0.085,0.153,0.344
random_negative,uniport_ppi_2017,0.818,0.182,0.172,0.157,0.321,0.427,0.610
random_negative,uniport_seq,0.661,0.339,0.198,0.077,0.165,0.229,0.384


In [48]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_cv_rank_save_auc0.8'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019, all_avg_df = show_table(root,ban_list=[],fnum=8,fused = True,input_weights = True)

ICD10_C50.csv not enough feature
ICD10_F90.csv not enough feature
ICD10_F72.csv not enough feature
ICD10_M41.csv not enough feature
ICD10_F01.csv not enough feature


method,para,auroc,rank_ratio,rank_ratio_std,bedroc_1,bedroc_5,bedroc_10,bedroc_30
random_negative,early_fusion,0.655,0.345,0.199,0.068,0.135,0.190,0.347
random_negative,geo_fused,0.831,0.170,0.166,0.172,0.326,0.421,0.603
random_negative,linear_fused,0.822,0.178,0.181,0.176,0.332,0.423,0.596
random_negative,ppi_2019_dw_40,0.842,0.158,0.150,0.153,0.333,0.433,0.614
random_negative,uniport_bio,0.739,0.261,0.199,0.096,0.222,0.300,0.466
random_negative,uniport_esm,0.655,0.345,0.199,0.068,0.135,0.190,0.347
random_negative,uniport_ppi_2019,0.827,0.174,0.191,0.158,0.326,0.426,0.614
random_negative,uniport_seq,0.679,0.321,0.221,0.081,0.176,0.237,0.393


In [5]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2017_cv_rank_save'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=[],fnum=10,fused = True,input_weights = True)

ICD10_F90.csv not enough feature


method,para,auroc,rank_ratio,rank_ratio_std,bedroc_1,bedroc_5,bedroc_10,bedroc_30
random_negative,early_fusion,0.648,0.352,0.192,0.051,0.133,0.196,0.356
random_negative,geo_fused,0.794,0.207,0.164,0.141,0.293,0.390,0.573
random_negative,linear_fused,0.802,0.198,0.173,0.152,0.307,0.407,0.587
random_negative,ppi_2017_dw_80,0.804,0.196,0.175,0.156,0.314,0.416,0.593
random_negative,uniport_esm,0.649,0.351,0.192,0.051,0.134,0.197,0.357
random_negative,uniport_exp,0.650,0.350,0.175,0.016,0.082,0.147,0.331
random_negative,uniport_ppi_2017,0.802,0.199,0.176,0.149,0.306,0.407,0.588
random_negative,uniport_seq,0.659,0.341,0.192,0.072,0.160,0.224,0.379
random_negative,weighted_geo_fused,0.791,0.209,0.165,0.136,0.291,0.389,0.569
random_negative,weighted_linear_fused,0.799,0.201,0.173,0.151,0.298,0.397,0.580


In [42]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_cv_rank_save'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=[],fnum=10,fused = True,input_weights = True)

ICD10_C50.csv not enough feature
ICD10_F72.csv not enough feature


method,para,auroc,rank_ratio,rank_ratio_std,bedroc_1,bedroc_5,bedroc_10,bedroc_30
random_negative,early_fusion,0.650,0.350,0.198,0.063,0.128,0.183,0.342
random_negative,geo_fused,0.817,0.183,0.164,0.151,0.294,0.388,0.574
random_negative,linear_fused,0.819,0.181,0.177,0.167,0.310,0.402,0.582
random_negative,ppi_2019_dw_40,0.839,0.161,0.150,0.143,0.319,0.421,0.607
random_negative,uniport_bio,0.738,0.262,0.195,0.090,0.211,0.290,0.461
random_negative,uniport_esm,0.651,0.349,0.197,0.063,0.129,0.184,0.343
random_negative,uniport_ppi_2019,0.817,0.184,0.190,0.148,0.307,0.404,0.592
random_negative,uniport_seq,0.675,0.325,0.217,0.077,0.173,0.235,0.391
random_negative,weighted_geo_fused,0.815,0.185,0.159,0.141,0.285,0.374,0.560
random_negative,weighted_linear_fused,0.817,0.183,0.174,0.157,0.298,0.390,0.572


In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2017_dw_test_auc'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=[],fused = True, input_weights = True)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_dw_test_auc'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=[],fused = True, input_weights = True)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2017_cv_norm'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=[],fused = True, input_weights = True)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2017_cv_norm'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=['D83','F01'],fused = True, input_weights = True)

In [ ]:
# root_disease = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_cv_update'
# for file in os.listdir(root_disease):
#     if file.startswith('ICD'):
#         print(file)
#         df = pd.read_csv(os.path.join(root_disease, file))
#         df['para'] = df['para'].str.replace(r'^later_weighted_rank', 'later_weighted_rank-', regex=True)
#         df['para'] = df['para'].str.replace(r'^later_avg_rank', 'later_avg_rank-', regex=True)
#         df.to_csv(os.path.join(root_disease, file), index=False)


In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_cv_nrom'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=[],fused = True, input_weights = True)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_cv_nrom'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=['N80','f01','D83'],fused = True, input_weights = True)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_cv_nrom'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=['f01','D83'],fused = True, input_weights = True)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_cv_bedroc_c'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=['f01','D83'],fused = True, input_weights = True)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2017_cv_update'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=[],fused = True, input_weights = True)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2017_cv_update'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=[],fnum=10, fused = True, input_weights = True)

In [ ]:
root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_cv_update'
# root = '/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_fused_geo_weight_bag_new_kernel_uni'

fused_2019 = show_table(root,ban_list=[],fused = True, input_weights = True)

In [ ]:
# full_fused_2019.to_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/single_weighted_fused_2019_bag.csv',index=False)

In [ ]:
# full_fused_2019=pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/single_weighted_fused_2019_bag.csv')

In [49]:
full_fused_2019 = fused_2019.round(3)
mapped_results = {icd: find_disease_category(icd) for icd in full_fused_2019['disease']}
full_fused_2019['category'] = full_fused_2019['disease'].map(mapped_results)
# full_fused_2019 = pd.read_csv('/itf-fi-ml/shared/users/ziyuzh/svm/results/weighted_fused_2019.csv')
full_fused_2019 = full_fused_2019[['method', 'para', 'auroc','rank_ratio','bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30', 'weights','disease', 'category']]
full_fused_2019['para'].unique()

array(['early_fusion', 'geo_fused', 'linear_fused', 'ppi_2019_dw_40',
       'uniport_bio', 'uniport_esm', 'uniport_ppi_2019', 'uniport_seq'],
      dtype=object)

In [50]:
# full_fused_2019['weights'] = full_fused_2019['weights'].astype(float)
full_fused_2019 = full_fused_2019.round(3)
# full_fused_2019 = full_fused_2019.rename(columns={'para': 'feature'})
# feature_order = ['DL']

feature_order = ['uniport_ppi_2019', 'ppi_2019_dw_40', 'uniport_bio', 'uniport_esm', 'uniport_seq', 
                 'linear_fused', 'geo_fused','early_fusion']

# feature_order = ['uniport_ppi_2019', 'ppi_2019_dw_40', 'uniport_bio', 'uniport_esm', 'uniport_seq', 
#                  'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused','early_fusion']

# feature_order = ['uniport_ppi_2017', 'ppi_2017_dw_80', 'uniport_exp', 'uniport_esm', 'uniport_seq', 
#                  'linear_fused', 'geo_fused','early_fusion']

# feature_order = ['uniport_ppi_2017', 'ppi_2017_dw_80', 'uniport_exp', 'uniport_esm', 'uniport_seq', 
#                  'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused','early_fusion']


# feature_order = ['uniport_ppi_2019', 'uniport_bio', 'uniport_esm', 'uniport_seq', 
#                  'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused','later_avg_rank', 'later_weighted_rank']

# feature_order = ['uniport_ppi_2017', 'uniport_exp', 'uniport_esm', 'uniport_seq', 
#                  'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused','later_avg_rank', 'later_weighted_rank']

# feature_order = ['uniport_ppi_2019', 'uniport_bio', 'uniport_esm', 'uniport_seq', 
#                  'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused']

# # Define the custom order for 'feature'
# feature_order = ['ppi_2019', 'bioconcept', 'esm2', 'uniport', 
#                  'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused']

# feature_order = ['ppi_2017', 'gene2vec', 'esm2', 'uniport', 
#                  'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused']

# Convert 'feature' to a categorical type with that order
full_fused_2019['para'] = pd.Categorical(full_fused_2019['para'], categories=feature_order, ordered=True)

# Now sort by 'disease' first, then 'feature' by the custom order
full_fused_2019 = full_fused_2019.sort_values(by=['disease', 'para'])

In [51]:
target_cols = ['auroc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30']

# Define alternating background colors per disease
base_colors = ['#ffdddd', '#dbf7db']
diseases = full_fused_2019['disease'].unique()
disease_color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(diseases)}

def combined_style(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)

    # Add background color row-wise
    for idx, row in df.iterrows():
        bg_color = disease_color_map[row['disease']]
        styles.loc[idx, :] = f'background-color: {bg_color};'

    # Highlight max in each group & each target column
    for category, group in df.groupby('disease'):
        for col in target_cols:
            max_val = group[col].max()
            max_indices = group[group[col] == max_val].index
            for idx in max_indices:
                styles.loc[idx, col] += ' color: red; font-weight: bold;'
    
    return styles

In [52]:
# Build mapping for background colors per disease
# full_fused_2019['para'] = full_fused_2019['para'].str.replace('ppi_2016', 'ppi_2017', regex=False)
# Identify numeric columns
numeric_cols = full_fused_2019.select_dtypes(include=['number']).columns

# Apply styling and format ONLY numeric columns
full_fused_2019_disease = (
    full_fused_2019.style
    .apply(combined_style, axis=None)
    .format({col: "{:.3f}" for col in numeric_cols})
)


full_fused_2019_disease

,method,para,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30,weights,disease,category
14,random_negative,uniport_ppi_2019,0.890,0.110,0.264,0.493,0.596,0.753,0.176,ICD10_C16,Neoplasms
11,random_negative,ppi_2019_dw_40,0.898,0.103,0.208,0.476,0.599,0.756,0.16,ICD10_C16,Neoplasms
12,random_negative,uniport_bio,0.812,0.189,0.089,0.244,0.351,0.570,0.288,ICD10_C16,Neoplasms
13,random_negative,uniport_esm,0.753,0.247,0.020,0.114,0.212,0.444,0.239,ICD10_C16,Neoplasms
15,random_negative,uniport_seq,0.714,0.287,0.011,0.098,0.179,0.394,0.248,ICD10_C16,Neoplasms
10,random_negative,linear_fused,0.888,0.113,0.201,0.443,0.567,0.734,0.166,ICD10_C16,Neoplasms
9,random_negative,geo_fused,0.891,0.110,0.143,0.411,0.561,0.744,0.185,ICD10_C16,Neoplasms
8,random_negative,early_fusion,0.753,0.247,0.020,0.114,0.212,0.444,0.239,ICD10_C16,Neoplasms
158,random_negative,uniport_ppi_2019,0.870,0.130,0.346,0.511,0.591,0.733,0.123,ICD10_C18,Neoplasms
155,random_negative,ppi_2019_dw_40,0.914,0.086,0.243,0.540,0.671,0.807,0.117,ICD10_C18,Neoplasms


In [ ]:
# abnormal = ['ICD10_C43','ICD10_N80','ICD10_N18', 'ICD10_G24', 'ICD10_E66', 'ICD10_D83', 'ICD10_C50']
# results = fused_2019[~fused_2019['disease'].isin(abnormal)]
# fused_2019

In [53]:
# fused_2019['para'] = fused_2019['para'].str.replace('ppi_2016', 'ppi_2017', regex=False)
results = fused_2019
mapped_results = {icd: find_disease_category(icd) for icd in results['disease']}
results['category'] = results['disease'].map(mapped_results)

collected_dfs = []
disease_num = dict()
for category in results['category'].unique().tolist():
    subdf = results[results['category'] == category].copy()
    subdf = subdf.drop(columns='weights')
    sum_df = create_summary(subdf, ['method', 'para'], fused=True)
    sum_df['category'] = category
    collected_dfs.append(sum_df)
    disease_num[category] = len(subdf) / 2

final_df = pd.concat(collected_dfs, ignore_index=True)

category_order = final_df.groupby('category', observed=True)['auroc_mean'].mean().sort_values(ascending=False).index.tolist()
final_df['category'] = pd.Categorical(final_df['category'], categories=category_order, ordered=True)

show_df = final_df.sort_values(by=['category', 'auroc_mean'], ascending=[True, False]) \
    .loc[:, final_df.columns.str.contains('method|para|auroc_mean|bedroc_1_mean|bedroc_5_mean|bedroc_10_mean|bedroc_30_mean|category', case=False)] \
    .round(3) \
    .rename(columns=lambda x: x.replace('_mean', ''))

# feature_order = ['ppi_2017', 'gene2vec', 'esm2', 'uniport', 
#                  'linear_fused', 'geo_fused','weighted_linear_fused','weighted_geo_fused']

# Convert 'feature' to a categorical type with that order
show_df['para'] = pd.Categorical(show_df['para'], categories=feature_order, ordered=True)

# Now sort by 'disease' first, then 'feature' by the custom order
show_df = show_df.sort_values(by=['category', 'para'])

# Define alternating background colors per disease
base_colors = ['#ffdddd', '#dbf7db']


# Build mapping for background colors per disease
category = show_df['category'].unique()
color_map = {disease: base_colors[i % len(base_colors)] for i, disease in enumerate(category)}



In [54]:
target_cols = ['auroc', 'bedroc_1', 'bedroc_5', 'bedroc_10', 'bedroc_30']

def combined_style2(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)

    # Add background color row-wise
    for idx, row in df.iterrows():
        bg_color = color_map[row['category']]
        styles.loc[idx, :] = f'background-color: {bg_color};'

    # Highlight max in each group & each target column
    for category, group in df.groupby('category'):
        for col in target_cols:
            max_val = group[col].max()
            max_indices = group[group[col] == max_val].index
            for idx in max_indices:
                styles.loc[idx, col] += ' color: red; font-weight: bold;'
    
    return styles


numeric_cols = show_df.select_dtypes(include=['number']).columns

# Apply styling and format ONLY numeric columns
styled_df = (
    show_df.style
    .apply(combined_style2, axis=None)
    .format({col: "{:.3f}" for col in numeric_cols})
)

styled_df
# styled_df.to_excel("/itf-fi-ml/shared/users/ziyuzh/svm/results/styled_2019_fused_weighted.xlsx", engine="openpyxl", index = False)

/tmp/ipykernel_2897310/597293719.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


,method,para,auroc,bedroc_1,bedroc_5,bedroc_10,bedroc_30,category
54,random_negative,uniport_ppi_2019,0.986,0.258,0.653,0.801,0.927,Diseases of the musculoskeletal system and connective tissue
51,random_negative,ppi_2019_dw_40,0.988,0.283,0.698,0.828,0.938,Diseases of the musculoskeletal system and connective tissue
52,random_negative,uniport_bio,0.974,0.217,0.559,0.709,0.878,Diseases of the musculoskeletal system and connective tissue
53,random_negative,uniport_esm,0.810,0.004,0.171,0.292,0.495,Diseases of the musculoskeletal system and connective tissue
55,random_negative,uniport_seq,0.736,0.021,0.262,0.362,0.481,Diseases of the musculoskeletal system and connective tissue
50,random_negative,linear_fused,0.993,0.455,0.805,0.893,0.963,Diseases of the musculoskeletal system and connective tissue
49,random_negative,geo_fused,0.991,0.429,0.778,0.877,0.956,Diseases of the musculoskeletal system and connective tissue
48,random_negative,early_fusion,0.810,0.004,0.171,0.292,0.495,Diseases of the musculoskeletal system and connective tissue
62,random_negative,uniport_ppi_2019,0.969,0.212,0.593,0.720,0.870,Diseases of the respiratory system
59,random_negative,ppi_2019_dw_40,0.969,0.295,0.606,0.723,0.869,Diseases of the respiratory system


In [55]:
all_avg_df = all_avg_df.set_index('para').loc[feature_order].reset_index()
def highlight_max_font_red(s):
    is_max = s == s.max()
    return ['color: red; font-weight: bold' if v else '' for v in is_max]
all_avg_df =all_avg_df[['para', 'auroc', 'rank_ratio', 'bedroc_1',
       'bedroc_5', 'bedroc_10', 'bedroc_30']].round(3)
# Apply styling
styled_all_avg_df = all_avg_df.style.apply(highlight_max_font_red, subset=target_cols)

# Display (in Jupyter or notebook environments)
styled_all_avg_df

,para,auroc,rank_ratio,bedroc_1,bedroc_5,bedroc_10,bedroc_30
0,uniport_ppi_2019,0.827000,0.174000,0.158000,0.326000,0.426000,0.614000
1,ppi_2019_dw_40,0.842000,0.158000,0.153000,0.333000,0.433000,0.614000
2,uniport_bio,0.739000,0.261000,0.096000,0.222000,0.300000,0.466000
3,uniport_esm,0.655000,0.345000,0.068000,0.135000,0.190000,0.347000
4,uniport_seq,0.679000,0.321000,0.081000,0.176000,0.237000,0.393000
5,linear_fused,0.822000,0.178000,0.176000,0.332000,0.423000,0.596000
6,geo_fused,0.831000,0.170000,0.172000,0.326000,0.421000,0.603000
7,early_fusion,0.655000,0.345000,0.068000,0.135000,0.190000,0.347000


In [56]:
with pd.ExcelWriter('/itf-fi-ml/shared/users/ziyuzh/svm/results/2019_dw_auc0.8.xlsx', engine='openpyxl') as writer:
    all_avg_df.to_excel(writer, sheet_name='all (macro avg)', index=False)
    styled_df.to_excel(writer, sheet_name='category (macro avg)', index=False)
    full_fused_2019_disease.to_excel(writer, sheet_name='disease', index=False)

/tmp/ipykernel_2897310/597293719.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for category, group in df.groupby('category'):


In [ ]:
### choose gamma
### entire dataset, get kernel (dist matrix)
### calculate nearest dist sum
### 2 * sum = gamma

In [ ]:
######## weighted kernel, weights comes from the biology pathway overlap
# 1. train / val split (no-time-cut) cross validation??
# 2. set threshold -200
# 3. pathway enrichment of predicted FPs and TPs
# 4. enriched reults similarity (jaccard-sim, Semantics similarity between terms)
# bootstrap bioconcept, gene2vec, uniport, esm2
##### fused only ppi and bioconcept
##### fused * bootstrap_bagging
##### fused 2017

In [ ]:
### hard negative
### low scores in disgenet
### perturb positive to be neg
### simialr disease positive to be positives